# Algoritmos Genéticos

## 1.1 La Naturaleza como Optimizadora

Los seres vivos son el resultado de miles de generaciones de experimentos evolutivos. Lo que vemos hoy en cualquier especie es la acumulación de pequeñas mejoras exitosas. Los experimentos fallidos desaparecieron porque los individuos resultantes no pudieron competir contra otros más aptos para sobrevivir.

El mecanismo central es la selección natural: los cambios heredados que confieren ventaja al individuo se transmiten a la descendencia. Con el tiempo, el modelo anterior es desplazado por el mejorado. Este proceso funciona porque:

- Las **mutaciones** generan variabilidad en la población.
- El **entorno** actúa como función de evaluación (¿qué tan apto es el individuo?).
- La **reproducción diferencial** amplifica los rasgos ventajosos.

## 1.2 Un Poco de Biología

### El Fenotipo y el Genotipo

Cada individuo posee un **fenotipo**: el conjunto de características observables (estatura, color de ojos, tipo sanguíneo). Estas características resultan de la interacción entre el entorno y la **herencia genética**. La información que determina el fenotipo está codificada en el **genoma**, contenido en los cromosomas de cada célula.

### Cromosomas, Genes y Alelos

Un **cromosoma** es una larga molécula de ADN formada por cuatro bases nucleotídicas: adenina (A), guanina (G), citosina (C) y timina (T). Las subcadenas de tres nucleótidos se llaman **codones**, y el conjunto de codones que codifica una proteína completa se denomina **gen**. El valor particular que toma un gen se llama **alelo**.

Las células *diploides* poseen dos juegos de cromosomas (uno del padre, uno de la madre). Si ambos juegos tienen el mismo alelo en un gen, el individuo es **homocigoto**; si difieren, es **heterocigoto**, y el alelo que se expresa es el **dominante**.

### Reproducción: Meiosis y Cruza

Las células reproductoras (gametos) no se replican por mitosis sino por **meiosis**, produciendo células *haploides* (un solo juego de cromosomas). Durante la meiosis ocurre la **cruza (crossover)**: dos cromosomas se parten en puntos llamados *quiasmas* e intercambian segmentos, generando cromosomas híbridos. Esto es la fuente principal de variabilidad genética.

### Mutación

La enzima **ADN-polimerasa** copia los cromosomas durante la replicación. Ocasionalmente comete errores (provocados por radiaciones o sustancias extrañas), alterando la secuencia original: esto es una **mutación**. La mayoría son desfavorables o letales, pero ocasionalmente otorgan ventaja al individuo mutante, quien la transmitirá a sus descendientes.

# Implementación: Clase Individuo y Clase Poblacion

In [1]:
import random

class Individuo:
    """
        * cromosoma: secuencia de genes (genotipo)
        * fenotipo: interpretación/decodificación del cromosoma
        * aptitud: qué tan bien se adapta al problema (función objetivo)
    """

    def __init__(self, longitud_cromosoma: int, cromosoma: list = None):
        """
        Inicializa un individuo
            * longitud_cromosoma: Número de genes en el cromosoma.
            * cromosoma: Lista de alelos. Si es None, se genera aleatoriamente con valores binarios (0 o 1), simulando los nucleótidos.
        """
        self.longitud_cromosoma = longitud_cromosoma

        # Genotipo: secuencia de alelos (genes con un valor asignado)
        if cromosoma is not None:
            self.cromosoma = cromosoma[:]
        else:
            self.cromosoma = [random.randint(0, 1) for _ in range(longitud_cromosoma)]

        # Fenotipo: interpretación del cromosoma (aquí, el valor entero)
        self.fenotipo = self._decodificar()

        # Aptitud: qué tan buena es esta solución (se calcula externamente)
        self.aptitud = 0.0


    #  ------------------ Métodos --------------------


    def _decodificar(self) -> int:
        """
        Convierte el genotipo (cadena de bits) en fenotipo (valor entero).
        Equivale a 'leer' qué proteína produce la secuencia de nucleótidos.
        """
        return int(''.join(map(str, self.cromosoma)), 2)

    def calcular_aptitud(self, funcion_objetivo) -> float:
        """
        Evalúa al individuo frente al entorno (función objetivo).
        Equivale al proceso de selección natural: el entorno 'juzga'
        qué tan apto es el organismo.

        Args:
            funcion_objetivo: Función f(fenotipo) -> float que mide la calidad.

        Returns:
            Valor de aptitud calculado.
        """
        self.aptitud = funcion_objetivo(self.fenotipo)
        return self.aptitud

    def mutar(self, tasa_mutacion: float = 0.01) -> None:
        """
        Aplica mutación puntual al cromosoma.
        Simula el error de la ADN-polimerasa al replicar: con baja probabilidad
        un gen cambia su alelo (0→1 o 1→0).

        Args:
            tasa_mutacion: Probabilidad de que cada gen mute (0.0 – 1.0).
        """
        for i in range(self.longitud_cromosoma):
            if random.random() < tasa_mutacion:
                self.cromosoma[i] = 1 - self.cromosoma[i]  # flip del bit
        self.fenotipo = self._decodificar()  # recalcula el fenotipo

    def __str__(self) -> str:
        cromosoma_str = ''.join(map(str, self.cromosoma))
        return (f"Individuo | cromosoma={cromosoma_str} "
                f"| fenotipo={self.fenotipo} | aptitud={self.aptitud:.4f}")

    def __repr__(self) -> str:
        return self.__str__()

In [2]:
class Poblacion:
    """
    Representa el conjunto de individuos que evolucionan a lo largo de generaciones sucesivas, simulando el proceso de selección natural.

    El ciclo evolutivo sigue:
        Inicialización -> Evaluación -> Selección -> Cruza -> Mutación -> (nueva generación)
    """

    def __init__(self,
                 tamano: int,
                 longitud_cromosoma: int,
                 tasa_mutacion: float = 0.01,
                 tasa_cruce: float = 0.7):
        """
        Inicializa la población con individuos generados aleatoriamente.

        Args:
            * tamano: Número de individuos en la población.
            * longitud_cromosoma: Longitud del cromosoma de cada individuo.
            * tasa_mutacion: Probabilidad de mutación por gen.
            * tasa_cruce: Probabilidad de que dos padres se crucen.
        """
        self.tamano = tamano
        self.longitud_cromosoma = longitud_cromosoma
        self.tasa_mutacion = tasa_mutacion
        self.tasa_cruce = tasa_cruce
        self.generacion = 0

        # Población inicial: individuos con cromosomas aleatorios
        self.individuos = [
            Individuo(longitud_cromosoma) for _ in range(tamano)
        ]

    #  ------------------ Métodos --------------------

    def evaluar(self, funcion_objetivo) -> None:
        """
        Evalúa la aptitud de todos los individuos.
        El entorno (función objetivo) actúa como presión selectiva.
        """
        for individuo in self.individuos:
            individuo.calcular_aptitud(funcion_objetivo)

    def seleccionar(self) -> 'Individuo':
        """
        Selecciona un individuo mediante torneo binario.
        Simula la competencia natural: de dos candidatos, sobrevive el más apto.

        Returns:
            El individuo ganador del torneo.
        """
        candidato_a = random.choice(self.individuos)
        candidato_b = random.choice(self.individuos)
        return candidato_a if candidato_a.aptitud >= candidato_b.aptitud else candidato_b

    def cruzar(self, padre: 'Individuo', madre: 'Individuo') -> tuple:
        """
        Aplica cruza en un punto (crossover), análoga a la meiosis.
        Se elige un quiasma (punto de corte) y se intercambian segmentos
        de los cromosomas de padre y madre, produciendo dos hijos híbridos.

        Args:
            padre: Primer progenitor.
            madre: Segundo progenitor.

        Returns:
            Tupla (hijo1, hijo2).
        """
        if random.random() < self.tasa_cruce:
            quiasma = random.randint(1, self.longitud_cromosoma - 1)
            cromo_hijo1 = padre.cromosoma[:quiasma] + madre.cromosoma[quiasma:]
            cromo_hijo2 = madre.cromosoma[:quiasma] + padre.cromosoma[quiasma:]
        else:
            # Sin cruza: los hijos son copias exactas de sus padres
            cromo_hijo1 = padre.cromosoma[:]
            cromo_hijo2 = madre.cromosoma[:]

        return (Individuo(self.longitud_cromosoma, cromo_hijo1),
                Individuo(self.longitud_cromosoma, cromo_hijo2))

    def evolucionar(self, funcion_objetivo) -> None:
        """
        Ejecuta un ciclo completo de evolución:

        Args:
            funcion_objetivo: Función que mide la aptitud de cada individuo.
        """
        self.evaluar(funcion_objetivo)

        nueva_generacion = []
        while len(nueva_generacion) < self.tamano:
            padre = self.seleccionar()
            madre = self.seleccionar()
            hijo1, hijo2 = self.cruzar(padre, madre)
            hijo1.mutar(self.tasa_mutacion)
            hijo2.mutar(self.tasa_mutacion)
            nueva_generacion.extend([hijo1, hijo2])

        self.individuos = nueva_generacion[:self.tamano]
        self.generacion += 1

    def get_mejor_individuo(self) -> 'Individuo':
        """
        Retorna al individuo con mayor aptitud en la generación actual.
        Equivale al organismo más adaptado al entorno.
        """
        return max(self.individuos, key=lambda ind: ind.aptitud)

    def get_aptitud_promedio(self) -> float:
        """Retorna la aptitud promedio de la población actual."""
        return sum(ind.aptitud for ind in self.individuos) / self.tamano

    def __str__(self) -> str:
        return (f"Poblacion | generacion={self.generacion} "
                f"| tamano={self.tamano} "
                f"| aptitud_promedio={self.get_aptitud_promedio():.4f}")

    def __repr__(self) -> str:
        return self.__str__()

## Demostración: Maximizar f(x) = x²

Se busca el valor de `x` (codificado en 8 bits) que maximice la función **f(x) = x²**.
El rango de x es [0, 255]. La solución óptima es x = 255 → f(255) = 65 025.

In [7]:
# Función objetivo: maximizar x^2
def funcion_objetivo(fenotipo: int) -> float:
    return fenotipo ** 2

# Configuración del experimento
TAMANO_POBLACION   = 20
LONGITUD_CROMOSOMA = 8   # 8 bits → x en [0, 255]
GENERACIONES       = 30
TASA_MUTACION      = 0.02
TASA_CRUCE         = 0.8

random.seed(42)
poblacion = Poblacion(TAMANO_POBLACION, LONGITUD_CROMOSOMA, TASA_MUTACION, TASA_CRUCE)

# --- Evaluar la población inicial (Gen 0) ---
poblacion.evaluar(funcion_objetivo)

print("=" * 60)
print(f"{'Gen':>5}  {'Mejor x':>8}  {'Mejor f(x)':>12}  {'Promedio f(x)':>14}")
print("=" * 60)

# Imprimir estado inicial
mejor_inicial = poblacion.get_mejor_individuo()
print(f"{poblacion.generacion:>5}  "
      f"{mejor_inicial.fenotipo:>8}  "
      f"{mejor_inicial.aptitud:>12.0f}  "
      f"{poblacion.get_aptitud_promedio():>14.0f}")

for _ in range(GENERACIONES):
    # 1. Evolucionar crea nuevos individuos (hijos)
    poblacion.evolucionar(funcion_objetivo)
    
    # 2. IMPORTANTE: Evaluar a los nuevos individuos
    # (Aunque evolucionar() ya llama a evaluar al inicio, 
    # necesitamos evaluar a los HIJOS antes de imprimir)
    poblacion.evaluar(funcion_objetivo)
    
    # 3. Ahora las estadísticas ya no serán 0
    mejor = poblacion.get_mejor_individuo()
    print(f"{poblacion.generacion:>5}  "
          f"{mejor.fenotipo:>8}  "
          f"{mejor.aptitud:>12.0f}  "
          f"{poblacion.get_aptitud_promedio():>14.0f}")

print("=" * 60)
print(f"\nMejor solución encontrada:")
print(f"  {mejor}")

  Gen   Mejor x    Mejor f(x)   Promedio f(x)
    0       240         57600           21844
    1       240         57600           31925
    2       243         59049           39825
    3       243         59049           49214
    4       243         59049           52023
    5       246         60516           54433
    6       247         61009           55405
    7       247         61009           58182
    8       247         61009           57273
    9       247         61009           59784
   10       247         61009           57365
   11       247         61009           59291
   12       247         61009           59481
   13       247         61009           57168
   14       247         61009           57768
   15       247         61009           60886
   16       247         61009           54895
   17       247         61009           56071
   18       255         65025           59785
   19       255         65025           60954
   20       255         65025     